In [ ]:
#Switch to python3.10 since it's required for tensorflow
import tensorflow as tf
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Tensorflow datasets


- In TensorFlow, a Dataset is a special data structure designed specifically for efficient data pipelines for machine learning. It is different from normal Python data structures like lists, arrays, or tensors because it is built to handle large-scale data loading, preprocessing, batching, and streaming efficiently.

In [ ]:
X = tf.range(10) # Create a Tensor with values from 0 to 9
dataset = tf.data.Dataset.from_tensor_slices(X) # This converts the tensor into a Dataset object where each element is one slice of the tensor.
for element in dataset:
    print(element.numpy()) # Print each element in the Dataset as a NumPy array

0
1
2
3
4
5
6
7
8
9


2026-03-06 21:34:41.084971: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


- **What is a Dataset in TensorFlow?**
    A Dataset is an object that represents a stream of data elements that can be:
    - loaded
    - transformed
    - shuffled
    - batched
    - prefetched 
    before feeding them to a machine learning model.
    Think of it as a data pipeline rather than just a container.


- **Why not use normal data structures?**
    Normal structures like:
    Python list
    NumPy array
    Tensor
    store the data but they don't manage how data flows to the model efficiently.
    Dataset does.


| Feature                | List / Array  | TensorFlow Dataset |
| ---------------------- | ------------- | ------------------ |
| Stores data            | ✔             | ✔                  |
| Handles huge datasets  | ❌ (needs RAM) | ✔                  |
| Lazy loading           | ❌             | ✔                  |
| Built-in batching      | ❌             | ✔                  |
| Parallel preprocessing | ❌             | ✔                  |
| Streaming from disk    | ❌             | ✔                  |


**Important concept: Lazy Execution**

A Dataset does not load everything immediately.
It loads data only when needed.
Example:

for item in dataset:
    print(item)

Now the dataset streams elements one by one.
This is crucial when training on datasets like:
ImageNet
huge text datasets
recommendation data
which cannot fit in RAM.

In [4]:
#Example of shuffling, batching and repeating a dataset
dataset = tf.data.Dataset.from_tensor_slices(X)
dataset = dataset.shuffle(10) # Shuffle the dataset with a buffer size of 10 (the size of the dataset). This means that the elements will be randomly shuffled.
dataset = dataset.batch(3) # Batch the dataset into batches of 3. This means that each element of the dataset will now be a batch of 3 elements from the original dataset.
dataset = dataset.repeat(2) # Repeat the dataset 2 times. This means that after going through all the batches once, it will start again from the beginning and go through all the batches a second time.
for batch in dataset:
    print(batch.numpy()) # Print each batch in the Dataset as a NumPy array

[1 5 9]
[2 4 6]
[8 0 7]
[3]
[6 8 0]
[1 4 7]
[3 9 5]
[2]


2026-03-06 21:42:53.816915: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


- **Why TensorFlow created tf.data.Dataset**

    Because deep learning training needs:
    - fast data loading
    - parallel preprocessing
    - GPU utilization
    - handling datasets larger than RAM


    Dataset solves all of this.

## Beginner-friendly pipeline using the California Housing dataset

In [31]:
housing = fetch_california_housing()
X = housing.data.astype("float32")
y = housing.target.astype("float32")

In [14]:
X.shape, y.shape

((20640, 8), (20640,))

In [32]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

In [29]:
"""Compute Mean and Standard Deviation
We will use these to standardize the data inside preprocess()."""
X_mean = X_train.mean(axis=0).astype("float32") # Calculate the mean of each feature (column) in the training set
X_std = X_train.std(axis=0).astype("float32") # Calculate the standard deviation of each feature (column) in the training set

X_mean = tf.constant(X_mean, dtype=tf.float32)
X_std = tf.constant(X_std, dtype=tf.float32)
print(X_mean)
print(X_std)

tf.Tensor(
[ 3.8807542e+00  2.8608284e+01  5.4352350e+00  1.0966847e+00
  1.4264530e+03  3.0969613e+00  3.5643150e+01 -1.1958229e+02], shape=(8,), dtype=float32)
tf.Tensor(
[1.9042363e+00 1.2602118e+01 2.3873026e+00 4.3320143e-01 1.1370220e+03
 1.1578394e+01 2.1366005e+00 2.0055928e+00], shape=(8,), dtype=float32)


In [ ]:
# Create a TensorFlow Dataset from the training data
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))


<_TensorSliceDataset element_spec=(TensorSpec(shape=(8,), dtype=tf.float64, name=None), TensorSpec(shape=(), dtype=tf.float64, name=None))>


Each element in the dataset is:

(features, label)

Example: ([longitude, latitude, ... median_income], house_price)

In [27]:
# Define a preprocessing function to standardize the features and expand the dimensions of the target variable
def preprocess(x, y):
    x = (x - X_mean) / X_std
    return x, y

In [ ]:
# Final pipeline for the training dataset, we will shuffle the data, batch it and prefetch it for better performance.
# train_ds = (
#     tf.data.Dataset.from_tensor_slices((X_train, y_train))
#     .map(preprocess)
#     .shuffle(10000)
#     .batch(32) # Batch the dataset into batches of 32. This means that each element of the dataset will now be a batch of 32 elements from the original dataset.
#     .prefetch(tf.data.AUTOTUNE) # This allows the dataset to fetch batches in the background while the model is training, improving performance.
# )

TypeError: in user code:

    File "/var/folders/mv/bm2sqsfj6ddf3cqcxqwbwpsw0000gn/T/ipykernel_97688/850102887.py", line 3, in preprocess  *
        x = (x - X_mean) / X_std

    TypeError: Input 'y' of 'Sub' Op has type float32 that does not match type float64 of argument 'x'.


In [33]:
# Full optimized pipeline
def create_dataset(X, y, training=True, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.map(preprocess)
    ds.cache() # Cache the dataset in memory after the first epoch. This means that after the first time the dataset is processed, it will be stored in memory and subsequent epochs will use the cached data, improving performance.
    if training:
        ds = ds.shuffle(len(X))
    
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE) #AuTOTUNE allows TensorFlow to determine the optimal number of batches to prefetch based on the system's resources and the workload.
    
    return ds

train_ds = create_dataset(X_train, y_train, training=True)
val_ds = create_dataset(X_val, y_val, training=False)
test_ds = create_dataset(X_test, y_test, training=False)

In [35]:
#Show first batch of the training dataset
for x_batch, y_batch in train_ds.take(1):
    print("Features (x_batch):", x_batch.numpy())
    print("Target (y_batch):", y_batch.numpy())

Features (x_batch): [[ 3.27977061e-01  4.27842081e-01  7.35838024e-04  9.58599672e-02
  -1.37598932e-01 -4.45364453e-02 -8.20532858e-01  6.29385471e-01]
 [ 9.48225617e-01 -6.03730619e-01  9.39362586e-01 -2.03949854e-01
  -4.61251438e-01 -5.10783829e-02  1.40262485e+00 -8.31529379e-01]
 [ 2.78567057e-02  5.86545527e-01 -2.18964741e-01 -6.12728059e-01
  -8.47347736e-01 -1.75509062e-02  9.34591711e-01 -1.24537301e+00]
 [-6.23743057e-01  6.65897310e-01 -4.84358191e-01 -7.29499534e-02
   4.48141724e-01 -9.47420895e-02 -1.35409045e+00  1.22272742e+00]
 [-1.88765511e-01  1.53876650e+00 -2.47209817e-01 -4.20447998e-02
  -3.70663911e-01 -9.28305238e-02 -5.67794502e-01 -6.36761561e-02]
 [-3.56024206e-01  2.69138575e-01 -5.10184407e-01  8.86719376e-02
   2.44099945e-01 -6.25045374e-02  1.00011611e+00 -1.44979942e+00]
 [ 1.28264427e-01  5.86545527e-01  6.37812167e-02 -2.44827613e-01
  -2.52812177e-01  3.99376452e-02 -7.31605411e-01  8.23841393e-01]
 [ 1.32302177e+00 -1.00048935e+00  2.91721046e-01

In [ ]:
#Create Neural Network model using Keras Sequential API
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation="relu", input_shape=(8,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1)
])

/Users/akash/cloudXlabs/cloudxlabs_mylearnings_projects/ml_notes/8_Datasets_tensorflow/.venv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [37]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [38]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 710us/step - loss: 0.8194 - mae: 0.6182 - val_loss: 0.4895 - val_mae: 0.4878
Epoch 2/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 593us/step - loss: 0.4097 - mae: 0.4546 - val_loss: 0.4009 - val_mae: 0.4555
Epoch 3/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 573us/step - loss: 0.3714 - mae: 0.4323 - val_loss: 0.3927 - val_mae: 0.4389
Epoch 4/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 556us/step - loss: 0.3539 - mae: 0.4200 - val_loss: 0.3776 - val_mae: 0.4346
Epoch 5/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 594us/step - loss: 0.3414 - mae: 0.4105 - val_loss: 0.3566 - val_mae: 0.4200
Epoch 6/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 556us/step - loss: 0.3292 - mae: 0.4034 - val_loss: 0.3637 - val_mae: 0.4325
Epoch 7/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 554us/step - loss: 0.3170 - mae: 0.3963 - val_loss: 0.3638 - val_mae: 0.4086
Epoch 8/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 579us/step - loss: 0.3144 - mae: 0.3922 - val_loss: 0.3508 - val_mae: 0.4209
Epoch 9/10
413/413 ━━━━━━━━━━━━━━━━━━━━ 

**Notice:
We are NOT passing numpy arrays anymore.
We are passing: train_ds**